# 第18课：Skill 封装与链式联动

本笔记本是课堂讲义。每个知识点包含：理论知识、案例代码、讲解、易错点与练习。综合练习 P1 使用课程根目录的教学运行时 [`agent_lab`](../agent_lab/README.md)。课后独立练习见 [chapter18_Skill封装与链式联动_课后练习.ipynb](chapter18_Skill封装与链式联动_课后练习.ipynb)。

**阶段定位**：阶段三 · 任务二 2.3 / 20分。20分

Skill 是业务动词：报价、下单、检索。它内部可以串起多个 Tool，并在失败时把错误变成下一轮输入。任务二 2.3 共 20 分。

## 学习目标

1. 基础：模型意图命中正确 Skill，入参符合 Pydantic。
2. 进阶：Tool A 产出经校验后注入 Tool B。
3. 高难：模糊指令先失败，再 Self-Correction 成功。

## 学习知识点

| 基础 6分 | 进阶 7分 | 高难 7分 |
| --- | --- | --- |
| 单工具精准命中 | 串行双工具 | 入参错误自愈 |
| WIDGET-X qty=1 | MINI qty=2 total=1398 | 小部件X → WIDGET-X |

## 基础回顾与案例提问

1. **R.1** 用户说“查一下 WIDGET-X 报价”，应调用 Skill 还是直接拼字符串？
2. **R.2** 库存工具返回的 sku 为什么要作为计价工具的输入，而不是重新从用户句子里抠？
3. **R.3** 自愈若第一轮就已经用别名成功，attempts 应是 1 还是 2？本课高难要求什么？

库存与单价以教学常量为准：X 价 1299 库存 12，MINI 价 699 库存 4。

使用 Python 3；需要 `pydantic`。从本课文件夹启动内核。本课不要求 GPU，也不强制安装 `langgraph` / `openai`。未配置私有化端点时，`get_client()` 返回进程内 Fake。不要使用 pandas。综合练习不要抄 `experiment.py` 的整段答案，按题面逐步完成。


In [ ]:
# R.1–R.3: Write and verify your predictions here.


In [ ]:
import sys
from pathlib import Path

COURSE = Path.cwd().resolve()
if COURSE.name.startswith("第"):
    COURSE = COURSE.parent
if str(COURSE) not in sys.path:
    sys.path.insert(0, str(COURSE))
print("已加入路径:", COURSE)
print("请从本课文件夹启动内核。未配置 OPENAI_BASE_URL 时使用教学 Fake 端点，不要求 GPU。")


## 1. Skill 是组合

### 理论知识

**Tool 做一件原子事，Skill 完成一笔业务。** quote_skill = 查库存 + 算金额。

### 案例：看函数签名


In [ ]:
from agent_lab.skills import quote_skill, self_correct_quote, normalize_sku
print("normalize_sku('小部件X') ->", normalize_sku("小部件X"))


### 讲解

别名表是教学用的最小清洗。真实项目会放在商品主数据，而不是写死在 Skill 里。

### 易错点与练习

1. **K1.1** 把别名清洗放在 Tool 内或 Skill 内各有何利弊？
2. **K1.2** MCP 的 kb_lookup 能否成为某个 Skill 的一步？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 2. 基础：单工具精准命中

### 理论知识

**入参完全符合规范：WIDGET-X、qty=1、price=1299。** 库存 12，total=1299。

### 案例：命中


In [ ]:
basic = quote_skill("WIDGET-X", qty=1, price=1299)
print(basic)
assert basic["ok"] and basic["sku"] == "WIDGET-X" 


### 讲解

6 分看的是意图到正确 SKU，而不是把所有输入都模糊匹配成功。

### 易错点与练习

1. **K2.1** qty 与 price 为何仍要 Pydantic？
2. **K2.2** 若模型把 MINI 说成 X，基础用例应否“聪明地改掉”？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 3. 进阶：双工具流转

### 理论知识

**A 的输出经过校验，再进 B。** 库存 ok 后才 calc_total。MINI×2×699=1398。

### 案例：串行


In [ ]:
chained = quote_skill("WIDGET-MINI", qty=2, price=699)
print(chained)
assert chained["ok"] and chained["total"] == 1398.0


### 讲解

不要在第二步重新输入用户的原始字符串。要用清洗后的 sku、qty、price。

### 易错点与练习

1. **K3.1** 若库存失败仍去计价，会制造什么假报表？
2. **K3.2** total 为什么不要在 Skill 里用 qty*price 自己乘一遍绕过 total_tool？本课要求走工具。

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 4. 数据清洗点

### 理论知识

**每一次跨工具都是 Schema 边界。** 非法 qty 应在 TotalArgs 失败，而不是算出负数。

### 案例：看失败形态（参数）


In [ ]:
from agent_lab.tools import total_tool
print(total_tool.run(qty=0, price=10))


### 讲解

Skill 要把这类 error 原样带到 QuoteResult.error，供下一轮模型阅读。

### 易错点与练习

1. **K4.1** 清洗发生在 Skill 还是 Tool？本课两处都有，如何分工？
2. **K4.2** 为什么 Result 同时保留 ok 与 error 空串？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 5. 高难：先失败

### 理论知识

**人为制造模糊指令。** “帮我查一下小部件X的库存报价” 不是合法 SKU。allow_repair=False 时必须失败。

### 案例：第一轮


In [ ]:
first = quote_skill("帮我查一下小部件X的库存报价", qty=2, price=1299, allow_repair=False)
print(first["ok"], first["error"])


### 讲解

first_error 应接近“未知 SKU”。没有这轮失败，自愈就变成普通别名，拿不到高难 7 分。

### 易错点与练习

1. **K5.1** 为何不在第一轮就 normalize？
2. **K5.2** 用户句子里混着“帮我查一下”对 Args.sku 的 min_length 意味着什么？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 6. Self-Correction

### 理论知识

**捕获错误原因，下一轮修正调用。** self_correct_quote 固定演示 attempts=2, repaired=True。

### 案例：第二轮


In [ ]:
repaired = self_correct_quote("帮我查一下小部件X的库存报价")
print(repaired)
assert repaired["ok"] and repaired["repaired"] is True and repaired["attempts"] == 2


### 讲解

qty 默认 2，price 默认 1299，总价 2598。这是教学约定，作业不要改常量还说“也对”。

### 易错点与练习

1. **K6.1** Agent 要把 first_error 放进下一轮 Prompt 的哪一角色消息里更合理？
2. **K6.2** 修正后仍失败应停止还是无限重试？对照 recursion_limit。

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 7. 状态保持

### 理论知识

**链式调用中的中间值要显式入状态。** 本课用返回 dict；第 14 课的 messages reducer 是同一思想。

### 案例：中间字段


In [ ]:
print({k: repaired[k] for k in ["sku", "stock", "qty", "total", "first_error"]})


### 讲解

评分时看这些字段是否对得上，而不是只看 ok=True。

### 易错点与练习

1. **K7.1** stock 来自哪一个 Tool？
2. **K7.2** first_error 在成功后为何仍保留？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 8. 20 分评分对照

### 理论知识

**6+7+7，三档都要交证据。**

### 案例：三连打印


In [ ]:
print("basic", basic["total"])
print("chained", chained["total"])
print("repaired", repaired["sku"], repaired["attempts"])


### 讲解

experiment.py 即此三连。综合练习请分格写。

### 易错点与练习

1. **K8.1** 只做自愈不做双工具，最高多少分？
2. **K8.2** 把 attempts 写死为 2 而不调用两轮，算不算造假？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 综合练习：三级 Skill

按 P1.1 → P1.2 → P1.3 顺序完成。每步都要能独立看出你做了什么。


### P1.1　基础命中

quote_skill WIDGET-X qty=1 price=1299，打印结果。


In [ ]:
# P1.1: basic hit.


### P1.2　双工具

WIDGET-MINI qty=2 price=699，确认 total=1398。


In [ ]:
# P1.2: chained tools.


### P1.3　自愈

self_correct_quote 模糊中文，确认 repaired 与 attempts。


In [ ]:
# P1.3: self-correct.


课后请打开 [chapter18_Skill封装与链式联动_课后练习.ipynb](chapter18_Skill封装与链式联动_课后练习.ipynb)。P1 基础、P2 进阶、P3 选做分析 first_error。
